# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name: 
Date: 

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/cloudnine_7/bootcamp_elrie_lin/homework/homework04

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [1]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? False


## Helpers (use or modify)

In [2]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [5]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function':'TIME_SERIES_DAILY_ADJUSTED','symbol':SYMBOL,'outputsize':'compact','apikey':os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js if 'Time Series' in k][0]
    df_api = pd.DataFrame(js[key]).T.reset_index().rename(columns={'index':'date','5. adjusted close':'adj_close'})[['date','adj_close']]
    df_api['date'] = pd.to_datetime(df_api['date']); df_api['adj_close'] = pd.to_numeric(df_api['adj_close'])
else:
    print("Alpha Vantage key not found/placeholder, using yfinance fallback...")
    import yfinance as yf
    df_api = yf.download(SYMBOL, period='3mo', interval='1d').reset_index()
    if isinstance(df_api.columns, pd.MultiIndex):
        df_api.columns = df_api.columns.get_level_values(0)
    target_col = 'Adj Close' if 'Adj Close' in df_api.columns else 'Close'
    df_api = df_api[['Date', target_col]].rename(columns={'Date': 'date', target_col: 'adj_close'})
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['adj_close'] = pd.to_numeric(df_api['adj_close'])

v_api = validate(df_api, ['date','adj_close']); v_api

[*********************100%***********************]  1 of 1 completed

Alpha Vantage key not found/placeholder, using yfinance fallback...


Price,date,adj_close
0,2026-05-26,308.064301
1,2026-05-27,310.582153
2,2026-05-28,312.240723
3,2026-05-29,311.791107
4,2026-06-01,306.046051
...,...,...
58,2026-08-18,310.029999
59,2026-08-19,316.829987
60,2026-08-20,311.299988
61,2026-08-21,309.350006


In [6]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data/raw/api_source-yfinance_symbol-AAPL_20260824-152534.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'  # TODO: replace with permitted page
headers = {'User-Agent':'AFE-Homework/1.0'}
try:
    print(f"Scraping table from {SCRAPE_URL}...")
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    # Locate the main constituents table
    table = soup.find('table', {'id': 'constituents'})
    if table:
        rows = [[c.get_text(strip=True) for c in tr.find_all(['th', 'td'])] for tr in table.find_all('tr')]
        header, *data = [r for r in rows if r]
        # 處理多餘的表頭與欄位對齊
        df_scrape = pd.DataFrame(data, columns=header)
        df_scrape = df_scrape[['Symbol', 'Security', 'GICS Sector', 'Date added']].copy()
    else:
        # 備用機制：使用 pandas read_html 解析
        tables = pd.read_html(resp.text)
        df_scrape = tables[0][['Symbol', 'Security', 'GICS Sector', 'Date added']].copy()
    df_scrape = df_scrape.rename(columns={
        'Symbol': 'symbol', 
        'Security': 'security', 
        'GICS Sector': 'sector', 
        'Date added': 'date_added'
    })
    #resp = requests.get(SCRAPE_URL, headers=headers, timeout=30); resp.raise_for_status()
    #soup = BeautifulSoup(resp.text, 'html.parser')
    #rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    #header, *data = [r for r in rows if r]
    #df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print('Scrape failed, using inline demo table:', e)
    html = '<table><tr><th>Ticker</th><th>Price</th></tr><tr><td>AAA</td><td>101.2</td></tr></table>'
    soup = BeautifulSoup(html, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)

#if 'Price' in df_scrape.columns:
    #df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')
#v_scrape = validate(df_scrape, list(df_scrape.columns)); v_scrape

#### Validation and Save
v_scrape = validate(df_scrape, ['symbol', 'security', 'sector'])
print("\nScrape Data Validation Results:", v_scrape)

In [17]:
# Part 2 — Web Scraping S&P 500 Table (Robust Header Matching)
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

print(f"Scraping table from {SCRAPE_URL}...")
resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, 'html.parser')
# 找到 S&P 500 主表格
table = soup.find('table', {'id': 'constituents'}) or soup.find('table', {'class': 'wikitable'})

# 提取所有資料列
rows = []
for tr in table.find_all('tr'):
    cols = [c.get_text(strip=True) for c in tr.find_all(['th', 'td'])]
    if cols:
        rows.append(cols)

header, *data = rows

# 轉成 DataFrame
df_raw = pd.DataFrame(data)

# 清理並尋找目標欄位索引 (避免大小寫、換行符或空格差異)
clean_headers = [str(h).strip().lower() for h in header]

sym_idx = next(i for i, h in enumerate(clean_headers) if 'symbol' in h or 'ticker' in h)
sec_idx = next(i for i, h in enumerate(clean_headers) if 'security' in h or 'company' in h)
sec_cat_idx = next(i for i, h in enumerate(clean_headers) if 'sector' in h)

# 擷取對應欄位並重新命名
df_scrape = pd.DataFrame({
    'symbol': df_raw[sym_idx],
    'security': df_raw[sec_idx],
    'sector': df_raw[sec_cat_idx]
})

# 驗證與存檔
v_scrape = validate(df_scrape, ['symbol', 'security', 'sector'])
print("\nScrape Data Validation Results:", v_scrape)
print(f"\nSuccessfully scraped {len(df_scrape)} records! Data Preview:")
print(df_scrape.head())

Scraping table from https://en.wikipedia.org/wiki/List_of_S%26P_500_companies...

Scrape Data Validation Results: {'missing': [], 'shape': (503, 3), 'na_total': 0}

Successfully scraped 503 records! Data Preview:
  symbol             security                  sector
0    MMM                   3M             Industrials
1    AOS          A. O. Smith             Industrials
2    ABT  Abbott Laboratories             Health Care
3   ABBV               AbbVie             Health Care
4    ACN            Accenture  Information Technology


In [18]:
_ = save_csv(df_scrape, prefix='scrape', site='wikipedia', table='sp500constituents')

Saved data/raw/scrape_site-wikipedia_table-sp500constituents_20260824-155057.csv


## Documentation
- API Source: Alpha Vantage / `yfinance` endpoint pulling daily historical prices for AAPL.
- Scrape Source: Wikipedia List of S&P 500 Companies (`https://en.wikipedia.org/wiki/List_of_S%26P_500_companies`).
- Assumptions & risks: 
  1. **Rate Limits**: Free API tier limits requests (e.g. 25 requests/day for Alpha Vantage); backoff strategy or fallback required.
  2. **Selector Fragility**: Web scraping depends on DOM elements (`#constituents`). Page layout or HTML changes will break parsing logic.
  3. **Data Integrity & Schema Changes**: Missing column validation guarantees schema stability before downstream processing.
  4. **Environment Isolation**: Local `.env` contains secrets and is excluded via `.gitignore`.
- Confirm `.env` is not committed.